# Spatially Masked Regression on AJILE12: intra-subject, cross-subject, masking and coverage

Ostadsharif Memar & Dehghani (2026), "Spatially Masked Regression Reveals Local and Distributed
Predictability in Electrophysiological Recordings", arXiv:2606.11415 (doi:10.48550/arxiv.2606.11415).
The claims file `claims/memar-2026.yaml` was written from the arXiv preprint, version 1, and its
code at commit 244dcdbd of github.com/neurovium/SpatiallyMaskedRegression.

Dataset: DANDI:000055 (AJILE12), version 0.220127.0436 (doi:10.48324/dandi.000055/0.220127.0436).
Why this version: the paper cites the dataset (Peterson et al. 2022) but no version of it. 0.220127.0436
is the only published version and the most recent; the draft (modified 2026-02-12) has the same
55 assets and size, and a draft can change. The files are streamed; nothing is written to disk.

This notebook tests the intra-subject and cross-subject claims. `masking_coverage.ipynb` tests the
mask sweep and the electrode coverage, and `surrogates_lagged.ipynb` the surrogate, lagged and
full-mask claims; each loads the data again. The EEG claims rest on a dataset the paper names without
an accession (BNCI Horizon 2020, Ofner et al. 2017) that is not in DANDI:000055 and is not tested.

| Claim | Here |
|---|---|
| intra-ieeg-mean | tested |
| table2-ieeg-intra-subjects | tested |
| cross-ieeg-mean | tested |
| table2-ieeg-cross-subjects | tested |
| cross-below-intra-ieeg | tested |
| fig4a-mask-sweep-ieeg, fig5a-coverage-ieeg | tested in `masking_coverage.ipynb` |
| lagged-ieeg-no-gain, surrogate-ieeg, surrogate-order-ieeg, full-mask-above-zero-ieeg | tested in `surrogates_lagged.ipynb` |
| intra-eeg-mean, intra-eeg-above-ieeg, cross-eeg-mean, transfer-penalty-larger-ieeg, lagged-eeg-no-gain, fig4b-mask-sweep-eeg, fig5b-coverage-eeg, surrogate-eeg, surrogate-order-eeg | blocked: the EEG data are not in this dataset (data elsewhere, no accession) |

Method details and where each comes from are in the docstring of `smr.py`. Two deviations, for
time, apply to every fit: the gradient steps and the validation loss use every 10th sample of
each trial (the test score uses all samples), and training stops after 200 epochs (the paper's
code allows 10000 with a patience of 200). `surrogates_lagged.ipynb` shows on one subject how
little either changes the score.

In [ ]:
import os, time, zlib, json, resource
import numpy as np
import pandas as pd
from scipy import stats
import smr

T0 = time.time()
PAPER = "memar-2026"
def seed(claim):
    return zlib.crc32(f"{PAPER}/{claim}".encode())

DECIM, MAX_EPOCH = 10, 200
SUBJECTS = [f"sub-{i:02d}" for i in range(1, 13)]
if os.environ.get("SMR_TEST"):                 # development only: two subjects
    SUBJECTS = ["sub-07", "sub-04"]
IDX = [int(s[-2:]) - 1 for s in SUBJECTS]
print("dataset", smr.DANDISET, smr.VERSION)

## Data check

Metadata of every file (no signal read), then the trials of every subject.

In [ ]:
sv = smr.survey()
pd.set_option("display.width", 250, "display.max_columns", 30, "display.max_rows", 100)
print(sv.drop(columns=["subject", "times"]).to_string(index=False))
print()
print("files", len(sv), "subjects", sv.subject.nunique(), "sessions per subject",
      sv.groupby("subject").size().to_dict())
print("sampling rates", sorted(sv.rate.unique()), "units", sorted(sv.unit.unique()),
      "conversion", sorted(sv.conversion.unique()))
print("channels per subject", sv.groupby("subject").channels.agg(["min", "max"]).to_dict("index"))
print("location labels in the electrode tables", sorted(set(",".join(sv.locations).split(","))))
print("files whose reaches table and ReachEvents timestamps differ by more than 10 ms:",
      sv.set_index("path").reach_vs_events_max_diff_s[sv.reach_vs_events_max_diff_s.values > 0.01].to_dict(),
      "; largest difference elsewhere (s):", sv.reach_vs_events_max_diff_s[sv.reach_vs_events_max_diff_s <= 0.01].max())
print("reach events after the end of the recording:", sv.set_index("path").reaches_after_end[sv.reaches_after_end.values > 0].to_dict())
print()
print("Everything the first file holds:")
print("\n".join(smr.list_file(smr.assets()[0]["asset_id"])))
print("total reach events", int(sv.reaches.sum()))

In [ ]:
t = time.time()
DATA = {}
for s in SUBJECTS:
    S = smr.load_subject(s)
    DATA[s] = smr.drop_uncoordinated(S)
    print(s, "trials", DATA[s]["a"].shape[0], "electrodes", DATA[s]["a"].shape[1],
          "dropped without coordinates", DATA[s]["n_dropped_el"],
          "events dropped at file edges", sum(x["dropped_edge"] for x in S["sessions"]),
          "dropped for NaN in the padded window", sum(x["dropped_nan"] for x in S["sessions"]),
          "of which NaN in the 2 s epoch", sum(x["nan_in_epoch"] for x in S["sessions"]),
          f"({time.time() - t:.0f} s)", flush=True)
print("peak memory MB", resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e3)

In [ ]:
rows = []
for s, S in DATA.items():
    xa = S["a"]
    rows.append(dict(subject=s, trials=xa.shape[0], electrodes=xa.shape[1], samples=xa.shape[2],
                     a_min=float(xa.min()), a_max=float(xa.max()), a_sd=float(xa.std()),
                     b_min=float(S["b"].min()), b_max=float(S["b"].max()),
                     hemisphere_x_neg=int((S["xyz"][:, 0] < 0).sum()), x_pos=int((S["xyz"][:, 0] > 0).sum()),
                     good_false=int((~S["el"]["good"]).sum()),
                     sessions=len(S["sessions"])))
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print("trials in all:", summary.trials.sum())

Duplicate recordings: each pair of sessions (within and across subjects with the same channel
count) is compared by its reach event times and by the per-channel SD of its trials. The clock:
high-gamma power after the event against before it, against a null with the labels swapped per
trial (protocol, author step 2).

In [ ]:
# per session: the channel correlation matrix of the pooled trials (reading a)
fp = []
for s, S in DATA.items():
    start = 0
    for x in S["sessions"]:
        n = x["trials"]
        v = S["a"][start:start + n].transpose(1, 0, 2).reshape(S["a"].shape[1], -1)
        fp.append((x["path"], np.corrcoef(v)))
        start += n
dups = [(fp[i][0], fp[j][0]) for i in range(len(fp)) for j in range(i + 1, len(fp))
        if fp[i][1].shape == fp[j][1].shape and np.allclose(fp[i][1], fp[j][1], atol=1e-3)]
times = dict(zip(sv.path, sv.times))
same_t = [(p, q) for i, p in enumerate(times) for q in list(times)[i + 1:]
          if len(times[p]) == len(times[q]) and np.allclose(times[p], times[q])]
print("session pairs with the same channel correlation matrix:", len(dups), dups)
print("session pairs with identical event times:", len(same_t), same_t)

In [ ]:
rng = np.random.default_rng(seed("intra-ieeg-mean") + 2)   # third generator of the claim, for the clock check
clock = []
for s, S in DATA.items():
    obs, null95, n4, _ = smr.clock_check(S["a"], rng)
    clock.append(dict(subject=s, max_abs_t=round(obs, 2), null_95=round(null95, 2), electrodes_abs_t_over_4=n4))
print(pd.DataFrame(clock).to_string(index=False))

## intra-ieeg-mean

> In the intra-subject setting, SMR reconstructs held-out iEEG with a mean distance correlation of
> 0.553 ± 0.068 across the 12 AJILE12 subjects.

The common pipeline at m = 1.0 (the whole 9-electrode neighbourhood masked with the target), under
both readings of where the preprocessing is applied. The atlas neighbourhood variant cannot be
run: every electrode's `location` is "unknown" (data check above), and the dataset holds no AAL
labels.

In [ ]:
def run_all(m, rng, reading="a", **kw):
    # one SMR condition on every subject, in order, from one generator
    out = {}
    for s in SUBJECTS:
        S = DATA[s]
        out[s] = smr.run_smr(S[reading].astype(np.float32), smr.neighbours(S["xyz"]), m, rng,
                             decim=DECIM, max_epoch=MAX_EPOCH, **kw)
    return out

def subj(res):
    return np.array([res[s]["dc"].mean() for s in SUBJECTS])

def msd(v):
    return f"{np.mean(v):.3f} ± {np.std(v):.3f}"

t = time.time()
R1a = run_all(1.0, np.random.default_rng(seed("intra-ieeg-mean")), "a")
print(f"reading a done ({time.time() - t:.0f} s)", flush=True)
R1b = run_all(1.0, np.random.default_rng(seed("intra-ieeg-mean") + 1), "b")
print(f"reading b done ({time.time() - t:.0f} s)", flush=True)

In [ ]:
TAB_INTRA = np.array([0.585, 0.647, 0.535, 0.452, 0.524, 0.428, 0.531, 0.487, 0.635, 0.605, 0.612, 0.593])[IDX]
TAB_INTRA_SD = np.array([0.041, 0.038, 0.035, 0.036, 0.034, 0.021, 0.028, 0.025, 0.041, 0.040, 0.032, 0.032])[IDX]
ia, ib = subj(R1a), subj(R1b)
df = pd.DataFrame({"subject": SUBJECTS, "electrodes": [len(R1a[s]["dc"]) for s in SUBJECTS],
                   "reading_a": ia.round(3), "sd_el_a": [R1a[s]["dc"].std().round(3) for s in SUBJECTS],
                   "reading_b": ib.round(3), "paper": TAB_INTRA,
                   "min_el_a": [R1a[s]["dc"].min().round(3) for s in SUBJECTS],
                   "max_el_a": [R1a[s]["dc"].max().round(3) for s in SUBJECTS]})
print(df.to_string(index=False))
for name, v in (("reading a (preprocess, then segment)", ia), ("reading b (preprocess each epoch)", ib)):
    print(f"{name}: mean over subjects {msd(v)}; within 0.553 ± 0.068: {abs(v.mean() - 0.553) <= 0.068}")
print("epochs run per subject (reading a):", [R1a[s]["n_ep"] for s in SUBJECTS])
print("median epoch of the best validation loss per subject:", [int(np.median(R1a[s]["best_ep"])) for s in SUBJECTS])

Seed sensitivity of intra-ieeg-mean (protocol, author step 3): the m = 1.0 run of reading (a) under
the claim's seed plus 1000, 2000, 3000 and 4000. Each draws the initial weights and the batch
order, and its seed also sets the 64/16/20 split (in place of 42).

In [ ]:
t = time.time()
SEEDS = {0: ia}
for add in (1000, 2000, 3000, 4000):
    rng = np.random.default_rng(seed("intra-ieeg-mean") + add)
    SEEDS[add] = subj(run_all(1.0, rng, "a", split_seed=int(seed("intra-ieeg-mean") + add) % 2**32))
    print(f"seed + {add}: mean over subjects {msd(SEEDS[add])} ({time.time() - t:.0f} s)", flush=True)
means = [v.mean() for v in SEEDS.values()]
print(f"range of the mean over the five seeds: {min(means):.3f} to {max(means):.3f}; all within 0.553 ± 0.068: "
      f"{all(abs(m - 0.553) <= 0.068 for m in means)}")
print("per subject range over seeds:", np.ptp(np.array(list(SEEDS.values())), axis=0).round(3).tolist())

**Result: reproduced.** The mean over the 12 subjects is 0.602 ± 0.058 under reading (a) and 0.570 ± 0.066 under reading (b), both within 0.553 ± 0.068; over five seeds the reading (a) mean ranges from 0.598 to 0.602.

## table2-ieeg-intra-subjects

> Intra-subject iEEG DistCorr differs across subjects, from 0.428 to 0.647, with the subject values
> Table II and Figure 2A give.

Row n of Table II is taken as sub-n (claims file). Each subject's value is compared with the
table's value, with the table's SD across electrodes as tolerance.

In [ ]:
for name, v in (("reading a", ia), ("reading b", ib)):
    within = np.abs(v - TAB_INTRA) <= TAB_INTRA_SD
    rho, p = stats.spearmanr(v, TAB_INTRA)
    print(f"{name}: {within.sum()} of 12 within tolerance; subjects within: "
          f"{[s for s, w in zip(SUBJECTS, within) if w]}; Spearman rho with Table II {rho:.2f} (p = {p:.3f}); "
          f"range {v.min():.3f} to {v.max():.3f}")
    print("   difference from the table:", np.round(v - TAB_INTRA, 3).tolist())

**Result: partially reproduced.** Under reading (b) 8 of 12 subject values are within the table's SD (Spearman rho 0.78 with Table II), under reading (a) 4 of 12 (rho 0.71); the values range from 0.431 to 0.669 (b) and 0.501 to 0.718 (a), against 0.428 to 0.647 in the paper.

## cross-ieeg-mean

> Transferring learned inter-electrode matrices from other subjects, through correlation-based
> electrode mapping and per-electrode donor selection, reconstructs a target subject's
> trial-averaged iEEG with a mean DistCorr of 0.389 ± 0.064, without fitting weights on the target.

Donor matrices are the m = 1.0 intra-subject weights of reading (a). The target's trials enter as
stored after preprocessing (not z-scored across trials), as the paper's evaluation script averages
the stored trials (Test_new_subject_iEEG.py, lines 101 to 106: split 50/50 with random_state 42,
mean over trials, z-score of each mean trace). Readings: which of the target's trials enter C (all,
as the code; or the validation half), and how C is computed (Eq. 3, paired trials; or the code's
concatenated series). Electrodes with no matched donor are left out of the mean.

In [ ]:
TAB_CROSS = np.array([0.331, 0.483, 0.440, 0.325, 0.509, 0.285, 0.415, 0.377, 0.409, 0.363, 0.395, 0.340])[IDX]
TAB_CROSS_SD = np.array([0.007, 0.013, 0.012, 0.006, 0.016, 0.004, 0.010, 0.013, 0.010, 0.007, 0.014, 0.006])[IDX]
t = time.time()
CROSS = {}
for which_c in ("all", "val"):
    for corr in ("paired", "concat"):
        vals, unmatched = [], []
        for s in SUBJECTS:
            donors = [(DATA[d]["a"], R1a[d]["W"]) for d in SUBJECTS if d != s]
            dc, choice = smr.cross_subject(DATA[s]["a"], donors, which_c=which_c, corr=corr)
            vals.append(np.nanmean(dc))
            unmatched.append(int(np.isnan(dc).sum()))
        CROSS[(which_c, corr)] = np.array(vals)
        v = CROSS[(which_c, corr)]
        print(f"C from {which_c} trials, {corr}: mean {msd(v)}; within 0.389 ± 0.064: {abs(v.mean() - 0.389) <= 0.064}; "
              f"electrodes without a donor per subject {unmatched} ({time.time() - t:.0f} s)", flush=True)
print(pd.DataFrame({"subject": SUBJECTS, **{f"{a}/{b}": v.round(3) for (a, b), v in CROSS.items()},
                    "paper": TAB_CROSS}).to_string(index=False))

**Result: reproduced.** The cross-subject mean is 0.351 ± 0.105 with C from all trials and Eq. 3, and 0.339 to 0.350 under the other three readings, all within 0.389 ± 0.064; every electrode got a donor.

## table2-ieeg-cross-subjects

> Cross-subject iEEG DistCorr ranges over subjects from 0.285 to 0.509, with the subject values
> Table II and Figure 2A give.

In [ ]:
for key, v in CROSS.items():
    within = np.abs(v - TAB_CROSS) <= TAB_CROSS_SD
    rho, p = stats.spearmanr(v, TAB_CROSS)
    print(f"{key}: {within.sum()} of 12 within tolerance {[s for s, w in zip(SUBJECTS, within) if w]}; "
          f"Spearman rho {rho:.2f} (p = {p:.3f}); range {v.min():.3f} to {v.max():.3f}")

**Result: not reproduced.** No subject value is within the table's SD under any reading (0 of 12), and the Spearman rho with Table II is 0.16 to 0.28; sub-12 is 0.647 against 0.340 in the table and sub-05 0.269 against 0.509.

## cross-below-intra-ieeg

> Cross-subject iEEG performance is lower than intra-subject performance but keeps appreciable
> predictive power.

In [ ]:
for key, v in CROSS.items():
    print(f"{key}: cross {v.mean():.3f} against intra (reading a) {ia.mean():.3f}: below {v.mean() < ia.mean()}; "
          f"subjects with cross below intra {int((v < ia).sum())} of 12; "
          f"cross-ieeg-mean within tolerance {abs(v.mean() - 0.389) <= 0.064}")

**Result: reproduced.** The cross-subject average (0.351) is below the intra-subject average (0.602), cross-ieeg-mean is reproduced, and 11 of 12 subjects (12 of 12 under the concatenated readings) have cross below intra.

## Summary

| Claim | Outcome |
|---|---|
| intra-ieeg-mean | reproduced (0.602 ± 0.058, reading a; 0.570 ± 0.066, reading b) |
| table2-ieeg-intra-subjects | partially reproduced (8 of 12 within tolerance, reading b) |
| cross-ieeg-mean | reproduced (0.351 ± 0.105) |
| table2-ieeg-cross-subjects | not reproduced (0 of 12) |
| cross-below-intra-ieeg | reproduced |
| fig4a-mask-sweep-ieeg, fig5a-coverage-ieeg | see `masking_coverage.ipynb` (reproduced; not reproduced) |
| surrogate-ieeg, surrogate-order-ieeg, full-mask-above-zero-ieeg, lagged-ieeg-no-gain | see `surrogates_lagged.ipynb` (reproduced; not reproduced; reproduced; reproduced) |
| the nine EEG claims | blocked: their data are not in DANDI:000055 and were not tested in this run |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: the trials are 2 s windows around every wrist movement
event in the `reaches` table, all sessions pooled (the paper does not say which events or days);
preprocessing over the whole recording is approximated by filtering each event with 5 s of padding
and taking the |z| and z-score statistics over the session's event windows; the gradient steps use
every 10th sample and training stops at 200 epochs (on sub-07 this changes the mean score by at
most 0.0003, `surrogates_lagged.ipynb`); electrodes without coordinates are left out (8 in sub-12);
events whose padded window holds NaN samples or runs past the recording are left out (7718 of 8088
trials kept).

What was found about the dataset: no electrode has an anatomical label (`location` is "unknown"
everywhere), so the code's atlas neighbourhood cannot be built; sub-12 has 8 electrodes without
coordinates; in four files the `reaches` table and the ReachEvents timestamps disagree by 29240 to
86400 s, and in three of them (the recordings shorter than 24 h) the table puts 22 to 132 events
after the end of the recording; sub-09, sub-06 and sub-12 have 16, 1 and 4 electrodes at x > 0 in
a left-hemisphere implant; sub-02 and sub-09 show no event-locked high-gamma change, so the event
clock is not established for them. No session was deposited twice.


In [ ]:
print(f"notebook ran {(time.time() - T0) / 60:.1f} min; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")